# Introduction to PyTorch

This notebook introduces the fundamental concepts of **PyTorch**, the deep learning framework we will use throughout this course.

### Learning Objectives
By the end of this notebook, you should be able to:
1. Create and manipulate **tensors** (PyTorch's core data structure).
2. Understand how PyTorch handles **devices** (CPU vs. GPU).
3. Use PyTorch's **automatic differentiation** (autograd) to compute gradients.
4. Build a simple neural network using both raw tensors and `torch.nn`.
5. Train a network using `torch.optim`.

## 1. Tensors

A **tensor** is a multidimensional array optimized for GPU computation. It is the fundamental data structure in PyTorch — analogous to NumPy's `ndarray`, but with two key additions:
- It can run on **GPUs** for accelerated computation.
- It supports **automatic differentiation** for gradient-based optimization.

### 1.1 Creating tensors from data

Let's start by loading an image and converting it to a tensor.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

print(f"PyTorch version: {torch.__version__}")

In [ ]:
# Load an image using PIL (you can replace with your own image)
input_image = np.array(Image.open("./images/bird.png"))
print(f"NumPy array — shape: {input_image.shape}, dtype: {input_image.dtype}")

plt.imshow(input_image)
plt.axis('off')
plt.title("Original image")
plt.show()

In [ ]:
# Convert to a PyTorch tensor
image_tensor = torch.tensor(input_image)
print(f"Tensor — shape: {image_tensor.shape}, dtype: {image_tensor.dtype}")

### 1.2 Tensor layout for images in PyTorch

PyTorch expects image tensors in **CHW** format: `(channels, height, width)`.  
Most image libraries load images as **HWC**: `(height, width, channels)`.  
We use `permute` to rearrange the axes.

In [ ]:
# HWC -> CHW
image = image_tensor.permute(2, 0, 1)
print(f"After permute: {image.shape}  (channels, height, width)")

**Tip:** In practice, `torchvision.transforms.ToTensor()` does this conversion (and scales to [0,1]) automatically.

### 1.3 Common ways to create tensors

PyTorch provides many tensor constructors, similar to NumPy.

In [ ]:
# Zeros and ones
print("zeros:", torch.zeros(2, 3))
print("ones: ", torch.ones(2, 3))

In [ ]:
# Random tensors
print("Normal(0,1):", torch.randn(2, 3))
print("Uniform[0,1):", torch.rand(2, 3))
print("Integers [10,100):", torch.randint(10, 100, size=(2, 3)))

In [ ]:
# From a list (useful for kernels, small weight matrices, etc.)
Sobel_x = torch.tensor([[-1, 0, 1],
                         [-2, 0, 2],
                         [-1, 0, 1]], dtype=torch.float32)
print(f"Sobel kernel:\n{Sobel_x}")

### 1.4 Basic operations

In [ ]:
x = torch.tensor([[1, 2, 3],
                   [4, 5, 6]])

print("x      =", x)
print("x * 10 =", x * 10)       # element-wise multiplication
print("x + 10 =", x + 10)       # element-wise addition
print("x ** 2 =", x ** 2)       # element-wise power

### 1.5 Reshaping, squeezing, and unsqueezing

These operations change the **shape** of a tensor without changing its data.

In [ ]:
x = torch.arange(1, 7)  # [1, 2, 3, 4, 5, 6]
print(f"Original:        {x}  shape={x.shape}")

# reshape
y = x.reshape(2, 3)
print(f"reshape(2,3):\n{y}  shape={y.shape}")

# flatten back
print(f"reshape(-1):     {y.reshape(-1)}  shape={y.reshape(-1).shape}")

In [ ]:
# unsqueeze adds a dimension of size 1
x = torch.randn(3, 4)
print(f"x.shape = {x.shape}")

y = x.unsqueeze(0)          # add batch dimension
print(f"unsqueeze(0) -> {y.shape}")

z = y.squeeze(0)             # remove it
print(f"squeeze(0)   -> {z.shape}")

### 1.6 Inner product, matrix multiplication, and concatenation

In [ ]:
# Inner (dot) product
a = torch.tensor([1., 1., 0.])
b = torch.tensor([1., 0., 1.])
print(f"dot({a}, {b}) = {a.dot(b)}")

# Matrix multiplication
A = torch.tensor([[1., 2.], [3., 4.]])
v = torch.tensor([1., 1.])
print(f"A @ v = {A @ v}")         # matrix-vector
print(f"A @ A = \n{A @ A}")       # matrix-matrix

# Concatenation
x = torch.arange(6).reshape(2, 3)
print(f"\ncat along rows (axis=0):\n{torch.cat([x, x], dim=0)}")
print(f"cat along cols (axis=1):\n{torch.cat([x, x], dim=1)}")

## 2. Devices: CPU and GPU

Deep learning benefits enormously from GPU acceleration. PyTorch makes it easy to move tensors between devices.

In [ ]:
# Check GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Create a tensor on the chosen device
x = torch.randn(3, 3, device=device)
print(f"x is on: {x.device}")

# Move an existing tensor to a device
y = torch.ones(3, 3)
y = y.to(device)
print(f"y is on: {y.device}")

**Important:** All tensors involved in an operation must be on the **same device**. You will get an error if you try to add a CPU tensor to a GPU tensor.

## 3. Automatic Differentiation (Autograd)

PyTorch can automatically compute gradients of any computation with respect to its inputs. This is the engine behind backpropagation.

To enable gradient tracking, set `requires_grad=True` when creating a tensor.

In [ ]:
x = torch.tensor([[2., -1.],
                   [1.,  1.]], requires_grad=True)
print(f"x = {x}")

### Computing gradients

Consider $ y = \sum_{i} x^2_i $. The partial derivative with respect to each element is:

$$ \frac{\partial y}{\partial x_i} = 2\, x_i $$

PyTorch computes these gradients via `y.backward()`, and the result is stored in `x.grad`.

In [ ]:
y = x.pow(2).sum()
print(f"y = {y}")

y.backward()
print(f"x.grad = {x.grad}")
# Verify: grad should be 2*x
print(f"2 * x  = {2 * x}")

### `torch.no_grad()` and `detach()`

During **inference** (testing), we don't need gradients. Disabling them saves memory and speeds up computation.

In [ ]:
# no_grad context manager — disables gradient tracking
with torch.no_grad():
    z = x * 2
    print(f"z.requires_grad = {z.requires_grad}")  # False

# detach() — creates a tensor that shares data but has no gradient history
z = x.detach()
print(f"z.requires_grad = {z.requires_grad}")  # False

## 4. Backpropagation Example: Training a Simple Neural Network

Let's train a small network with **one hidden layer** (3 neurons, sigmoid activation) to map the input `[1, 1]` to the output `[0]`.

We will first do it using **raw tensors and autograd**, to understand what happens under the hood.

![Simple NN](./figs/Simple_NN.png)

### 4.1 Using raw tensors

In [ ]:
# Input and desired output
x = torch.tensor([[1., 1.]])
y_true = torch.tensor([[0.]])

# Weight initialization (fixed seed for reproducibility)
torch.manual_seed(42)
W1 = torch.randn(2, 3, requires_grad=True)   # input -> hidden (2 inputs, 3 hidden neurons)
b1 = torch.randn(1, 3, requires_grad=True)   # hidden bias
W2 = torch.randn(3, 1, requires_grad=True)   # hidden -> output
b2 = torch.randn(1, 1, requires_grad=True)   # output bias

print(f"W1 = {W1}")
print(f"b1 = {b1}")
print(f"W2 = {W2}")
print(f"b2 = {b2}")

In [ ]:
def feed_forward(x, y_true, W1, b1, W2, b2):
    """Forward pass: input -> hidden (sigmoid) -> output -> MSE loss."""
    hidden = torch.sigmoid(x @ W1 + b1)
    y_pred = hidden @ W2 + b2
    loss = torch.mean((y_pred - y_true) ** 2)
    return loss

def train_step(x, y_true, weights, lr):
    """One training step: forward, backward, update."""
    loss = feed_forward(x, y_true, *weights)
    loss.backward()
    
    # Update weights using gradient descent (no_grad to avoid tracking this operation)
    with torch.no_grad():
        updated = [w - lr * w.grad for w in weights]
    
    # Reset gradients and re-enable tracking
    updated = [w.clone().detach().requires_grad_(True) for w in updated]
    return updated, loss.item()

In [ ]:
# Training loop
weights = [W1, b1, W2, b2]
lr = 0.5
losses = []

for epoch in range(200):
    weights, loss = train_step(x, y_true, weights, lr)
    losses.append(loss)

plt.plot(losses)
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Training loss (raw tensors + autograd)")
plt.show()

In [ ]:
# Test the trained network
with torch.no_grad():
    hidden = torch.sigmoid(x @ weights[0] + weights[1])
    y_pred = hidden @ weights[2] + weights[3]
    print(f"Input: {x}")
    print(f"Predicted output: {y_pred.item():.6f}")
    print(f"Desired output:   {y_true.item():.1f}")

### 4.2 The same network using `torch.nn` and `torch.optim`

In practice, we don't build networks from raw tensors. PyTorch provides `torch.nn` for defining layers and models, and `torch.optim` for optimization algorithms.

This is the pattern you will use throughout the course.

In [ ]:
import torch.nn as nn
import torch.optim as optim

class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(2, 3)    # 2 inputs -> 3 hidden neurons
        self.output = nn.Linear(3, 1)    # 3 hidden -> 1 output
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        x = self.sigmoid(self.hidden(x))
        x = self.output(x)
        return x

model = SimpleNet()
print(model)

In [ ]:
# Define loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.5)

# Training loop
losses = []
for epoch in range(200):
    y_pred = model(x)                # forward pass
    loss = criterion(y_pred, y_true) # compute loss
    
    optimizer.zero_grad()            # reset gradients
    loss.backward()                  # compute gradients
    optimizer.step()                 # update weights
    
    losses.append(loss.item())

plt.plot(losses)
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Training loss (torch.nn + torch.optim)")
plt.show()

In [ ]:
# Test
model.eval()
with torch.no_grad():
    y_pred = model(x)
    print(f"Input: {x}")
    print(f"Predicted output: {y_pred.item():.6f}")
    print(f"Desired output:   {y_true.item():.1f}")

### Key takeaways

The `torch.nn` version follows a standard pattern that you will see in every PyTorch project:

1. **Define** the model as a subclass of `nn.Module`.
2. **Choose** a loss function (`nn.MSELoss`, `nn.CrossEntropyLoss`, ...).
3. **Choose** an optimizer (`optim.SGD`, `optim.Adam`, ...).
4. **Training loop:**
   - Forward pass → compute loss → `zero_grad()` → `backward()` → `step()`
5. **Evaluation:** switch to `model.eval()` and use `torch.no_grad()`.